# Merchy-AI Week 3: Chatbot Q&A evaluation (baseline vs RAG)

**Goal:** measure how accurately the support chatbot answers buyer questions, and whether retrieval (RAG) over the store's listings improves it.

**How to run:** top to bottom, from `backend/eval/` with the `.venv` kernel. Every Gemini result is saved under `backend/data/qa/`, so re-running a cell (or restarting Jupyter) never repeats finished work. If Gemini returns 503, wait a few minutes and run the same cell again.

| Part | What it does | Output |
|---|---|---|
| 1. Setup | paths, `.env`, Gemini client | |
| 2. Store facts | the policies answers are graded against | `store_facts_v1.json` |
| 3. Data | feedback + labels, cleaned listings | |
| 4. Test set | after-sale questions from complaints, pre-sale questions from listings, policy questions | `qa_test_set_v1.csv` |
| 5. Evaluation | answer, grade (Gemini as judge), log to MLflow, compare | `graded_*.csv`, `mlruns/` |

## 1. Setup

In [ ]:
import json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Find backend/ whether the notebook runs from backend/eval, backend or the repo root
BACKEND = Path.cwd()
if (BACKEND / 'backend').exists():
    BACKEND = BACKEND / 'backend'
while not (BACKEND / 'data').exists() and BACKEND != BACKEND.parent:
    BACKEND = BACKEND.parent
DATA = BACKEND / 'data'
QA = DATA / 'qa'
CACHE = QA / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(BACKEND / '.env', override=True)

API_KEY = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
assert API_KEY, 'Add GEMINI_API_KEY=... to backend/.env'
client = genai.Client(api_key=API_KEY)
MODEL = 'gemini-3.5-flash-lite'
EMBED_MODEL = 'gemini-embedding-001'
PAUSE = 4   # seconds between requests: stays under 15 requests/minute

def ask_gemini(system, payload, retries=4):
    for attempt in range(retries):
        try:
            resp = client.models.generate_content(
                model=MODEL, contents=json.dumps(payload, ensure_ascii=False),
                config={'system_instruction': system, 'temperature': 0,
                        'response_mime_type': 'application/json'})
            return json.loads(resp.text)
        except Exception as e:
            wait = 10 * 2 ** attempt
            print(f'  attempt {attempt + 1} failed ({str(e)[:60]}), retrying in {wait}s')
            time.sleep(wait)
    raise RuntimeError('Gemini is busy. Wait a few minutes and run this cell again; finished work is kept.')

def load_cache(name):
    path = CACHE / f'{name}.json'
    return json.loads(path.read_text()) if path.exists() else {}

def save_cache(name, data):
    (CACHE / f'{name}.json').write_text(json.dumps(data, indent=1, ensure_ascii=False))

print('Backend folder:', BACKEND)

## 2. Store facts

Answers are graded against these facts. Items marked `(assumed)` follow eBay platform rules and common practice for parts sellers; confirm them on a Paul Eagle listing (Shipping, returns & payments) and remove the tag. The tag is only for you: the chatbot and the judge see the text without it.

Comparing `facts-in-prompt` with `rag` is fair either way, because both use the same facts.

In [ ]:
STORE_FACTS = {
    'returns':       '(assumed) 30-day returns; buyer pays return shipping for change-of-mind returns. Items that are damaged, faulty or not as described can always be returned under the eBay Money Back Guarantee, and the seller pays return shipping.',
    'refunds':       '(assumed) refund issued through eBay within 2 business days after the return arrives and is checked',
    'handling_time': '(assumed) ships within 1 business day after payment',
    'carriers':      '(assumed) USPS Ground Advantage or UPS Ground; tracking number uploaded to eBay when the label is printed',
    'lost_package':  '(assumed) if tracking shows no movement for 7 days or the package is marked delivered but not received, contact us through eBay messages and we open a claim with the carrier',
    'fitment':       '(assumed) buyer should match the OEM part number in the listing; we can confirm fit if the buyer sends year, make and model before buying; if a part does not fit the vehicle listed as compatible, it is treated as not as described',
    'condition':     '(assumed) used parts are pulled from working machines and inspected; photos show the actual item; normal wear is described in the listing',
    'damaged_item':  '(assumed) send photos of the item and box within 7 days of delivery; we replace or refund',
    'contact':       'eBay messages; we typically respond within 24 hours',   # confirmed from listing descriptions
}
assumed = [k for k, v in STORE_FACTS.items() if v.startswith('(assumed)')]
FACTS = {k: v.replace('(assumed) ', '') for k, v in STORE_FACTS.items()}
FACTS_TEXT = '\n'.join(f'- {k}: {v}' for k, v in FACTS.items())
(QA / 'store_facts_v1.json').write_text(json.dumps(STORE_FACTS, indent=2))
print(FACTS_TEXT)
print(f'\nStill assumed, confirm with the store: {assumed}')

## 3. Data

**Feedback:** the cleaned comments of both stores, with the v1 Gemini labels (v2 if it exists). v1 labels some praise ("fast shipping") as shipping, so only comments the buyer rated negative or neutral are used for questions.

**Listings:** Paul Eagle's active listings from `fetch_listings.py`. Duplicate titles are dropped, and at most 5 listings may share the same description (one body kit was listed 100+ times).

In [ ]:
frames = []
for store, (folder, filename) in {'Paul Eagle': ('pauleagleauctions', 'feedback_clean_pauleagle.csv'),
                                  'fellers': ('fellerssmallengines', 'feedback_clean_fellers.csv')}.items():
    d = pd.read_csv(DATA / folder / filename, dtype={'feedback_id': str})
    d['store'] = store
    d['uid'] = store + '-' + d['feedback_id']
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

for LABELS_SOURCE in ['feedback_labels_v2.csv', 'feedback_labels_llm_both.csv']:
    if (DATA / LABELS_SOURCE).exists():
        break
labels = pd.read_csv(DATA / LABELS_SOURCE, dtype=str)
label_col = 'label' if 'label' in labels.columns else 'llm_label'
labels = labels[['uid', label_col]].drop_duplicates('uid').rename(columns={label_col: 'label'})
df = df.merge(labels, on='uid', how='inner')
print(len(df), 'labeled comments from', LABELS_SOURCE)

listings = pd.read_csv(QA / 'listings.csv', dtype=str).fillna('')
listings = listings.drop(columns=[c for c in ['condition_notes'] if c in listings.columns])
listings = listings.drop_duplicates('title')
listings = listings.groupby(listings['description'].str[:120], group_keys=False).head(5).reset_index(drop=True)
listings['text'] = (listings['title'] + '. Condition: ' + listings['condition'] + '. '
                    + listings['item_specifics'] + ' ' + listings['description']).str[:1200]
print(len(listings), 'listings after cleaning')

## 4. Test set

Three kinds of questions:

1. **After-sale** (need a person): 6 complaints per issue type, rewritten by Gemini as the message the buyer sent the seller *after* the order arrived. Answer key points come from the store facts.
2. **Pre-sale from listings** (chatbot should answer): 2 questions for each of 10 listings, with key points from the listing text and the store facts.
3. **Policy questions** (chatbot should answer): 8 written by hand from the store facts.

Results from earlier runs (`qa_test_set_draft.csv`, `qa_test_set_v1.csv`) are reused, so only missing questions call Gemini.

Lesson from building it: the first rewrite prompt said "before leaving feedback", and Gemini wrote pre-purchase questions instead of complaints. No error, only reading the output caught it, the same kind of silent prompt bug as v1 labeling.

In [ ]:
REWRITE_SYSTEM = '''You help build a test set for a customer-support chatbot of an eBay store that sells motorcycle and small-engine parts.
Each input is real buyer feedback left after a problem with an order.
Write the message that buyer sent the seller AFTER the order arrived (or failed to arrive), describing that same problem and asking the seller to fix it.
It must be about the problem in the feedback, never a question asked before buying.
Keep the details: the part, what went wrong, how long they waited. Remove names, order numbers and contact details.
Write like a real buyer: short and informal, one to three sentences.
Reply with only a JSON object mapping each id to the message.'''

KEY_SYSTEM = f'''You write the answer key for a customer-support test set of an eBay motorcycle and small-engine parts store.
Store facts (the only source of truth):
{FACTS_TEXT}

For each buyer question, give:
- key_points: 2 to 3 short points a correct reply must contain, based only on the store facts. If the facts do not answer it, one key point should be that staff will follow up.
- escalate: true if a person must act (refund, return label, damaged, wrong or missing item, lost package claim) or the facts do not cover the question; otherwise false.
Reply with only a JSON object mapping each id to {{"key_points": [...], "escalate": true or false}}.'''

complaints = df[(df['label'] != 'positive') & (df['comment_type'] != 'POSITIVE')
                & (df['comment_text'].str.len() >= 25)]
sample = complaints.sample(frac=1, random_state=42).groupby('label').head(6)

# Reuse questions and keys already written in earlier runs
rewrites, keys = load_cache('after_sale_rewrites'), load_cache('after_sale_keys')
for old in ['qa_test_set_draft.csv', 'qa_test_set_v1.csv']:
    if (QA / old).exists():
        o = pd.read_csv(QA / old, dtype=str).fillna('')
        for r in o[o['source'] == 'feedback'].itertuples():
            if r.source_id in set(sample['uid']) and r.question:
                rewrites.setdefault(r.source_id, r.question)
                if r.key_points:
                    keys.setdefault(r.source_id, {'key_points': [p.strip() for p in r.key_points.split('|') if p.strip()],
                                                  'escalate': r.escalate.lower() == 'true'})
save_cache('after_sale_rewrites', rewrites); save_cache('after_sale_keys', keys)

todo = sample[~sample['uid'].isin(rewrites)]
print(len(sample) - len(todo), 'rewrites reused,', len(todo), 'to write')
for start in range(0, len(todo), 15):
    part = todo.iloc[start:start + 15]
    rewrites.update(ask_gemini(REWRITE_SYSTEM, {r.uid: r.comment_text for r in part.itertuples()}))
    save_cache('after_sale_rewrites', rewrites); time.sleep(PAUSE)

todo = [u for u in sample['uid'] if u in rewrites and u not in keys]
print(len(todo), 'answer keys to write')
for start in range(0, len(todo), 10):
    part = todo[start:start + 10]
    keys.update(ask_gemini(KEY_SYSTEM, {u: rewrites[u] for u in part}))
    save_cache('after_sale_keys', keys); time.sleep(PAUSE)

after_sale = pd.DataFrame([{'qid': f'a{i:02d}', 'source': 'feedback', 'source_id': r.uid, 'category': r.label,
                            'source_text': r.comment_text, 'question': rewrites[r.uid],
                            'key_points': ' | '.join(keys[r.uid]['key_points']),
                            'escalate': bool(keys[r.uid]['escalate'])}
                           for i, r in enumerate(sample.itertuples()) if r.uid in keys])
print(len(after_sale), 'after-sale questions')
after_sale[['category', 'question', 'escalate']].head(5)

In [ ]:
PRESALE_SYSTEM = f'''You help build a test set for a customer-support chatbot of an eBay store that sells motorcycle and small-engine parts.
For each listing, write two short, realistic questions a buyer might send BEFORE buying: fit for a specific vehicle and year, condition, what is included, or shipping time. At least one question per listing must be answerable from the listing text.
For each question also give:
- key_points: 1 to 3 short points a correct reply must contain, using only the listing text and these store facts:
{FACTS_TEXT}
  If neither answers the question, the only key point is that staff will follow up.
- escalate: true only if neither the listing nor the store facts answer the question.
Reply with only JSON: {{"<listing id>": [{{"question": "...", "key_points": ["..."], "escalate": true or false}}]}}'''

presale = load_cache('pre_sale')
if not presale and (QA / 'qa_test_set_v1.csv').exists():   # reuse listing questions from the earlier run
    o = pd.read_csv(QA / 'qa_test_set_v1.csv', dtype=str).fillna('')
    presale = {'rows': o[o['source'] == 'listing'].to_dict('records')}
if not presale.get('rows'):
    picked = listings.sample(min(10, len(listings)), random_state=1)
    items = {f'listing-{i}': t for i, t in enumerate(picked['text'])}
    out = ask_gemini(PRESALE_SYSTEM, items)
    presale = {'rows': [{'source': 'listing', 'source_id': k, 'category': 'pre_sale', 'source_text': items[k],
                         'question': q['question'], 'key_points': ' | '.join(q['key_points']),
                         'escalate': bool(q['escalate'])}
                        for k, qs in out.items() if k in items for q in qs[:2]]}
save_cache('pre_sale', presale)

pre_sale = pd.DataFrame(presale['rows'])
pre_sale['qid'] = [f'l{i:02d}' for i in range(len(pre_sale))]
pre_sale['escalate'] = pre_sale['escalate'].astype(str).str.lower() == 'true'
print(len(pre_sale), 'pre-sale questions from listings')
pre_sale[['question', 'key_points', 'escalate']].head(4)

In [ ]:
POLICY_QUESTIONS = [
    ('How fast do you ship after I pay?', 'Ships within 1 business day after payment'),
    ('Will I get a tracking number?', 'Tracking number is uploaded to eBay when the label is printed'),
    ('Which carrier do you use?', 'USPS Ground Advantage or UPS Ground'),
    ("What's your return policy if I change my mind?", '30-day returns | Buyer pays return shipping for change-of-mind returns'),
    ('Can you tell me if this fits my 2015 Honda CRF250?', 'Match the OEM part number in the listing | Store can confirm fit if given year, make and model'),
    ('Is this used part tested before you sell it?', 'Used parts are pulled from working machines and inspected | Photos show the actual item'),
    ('If the part arrives damaged, what do I do?', 'Send photos of the item and box within 7 days of delivery | Store replaces or refunds'),
    ('How long does it take to get my refund after I send it back?', 'Refund issued through eBay within 2 business days after the return arrives and is checked'),
]
policy = pd.DataFrame([{'qid': f'p{i:02d}', 'source': 'manual', 'source_id': '', 'category': 'policy',
                        'source_text': '', 'question': q, 'key_points': k, 'escalate': False}
                       for i, (q, k) in enumerate(POLICY_QUESTIONS)])

test = pd.concat([after_sale, pre_sale, policy], ignore_index=True)
test['keep'] = 'Y'
cols = ['qid', 'source', 'source_id', 'category', 'source_text', 'question', 'key_points', 'escalate', 'keep']
test[cols].to_csv(QA / 'qa_test_set_v1.csv', index=False)
test['key_points'] = test['key_points'].map(lambda s: [p.strip() for p in s.split('|') if p.strip()])

print(len(test), 'questions saved to qa_test_set_v1.csv:', test['source'].value_counts().to_dict())
print('Needs a person:', test['escalate'].sum(), '| chatbot should answer:', (~test['escalate']).sum())

## 5. Evaluation

| Run | Model | How it gets store knowledge |
|---|---|---|
| `facts-in-prompt` | Gemini | all store facts in the prompt, no retrieval |
| `rag` | Gemini | for each question, the closest store facts and listings, found with Gemini embeddings |
| `baseline-real-chatbot` | the app's model | the app as it is today (optional: set `CHATBOT_URL`) |

`facts-in-prompt` and `rag` share the model and the prompt, so the gap between them is the effect of retrieval.

A reply is **correct** only if it covers every key point, escalates exactly when it should, and states nothing that is not in the store facts or the listing. **Automation rate** is the share of questions that don't need a person where the chatbot answered correctly by itself: that is where staff time is saved.

In [ ]:
import mlflow
# Newer MLflow needs a database backend; SQLite is a single local file
mlflow.set_tracking_uri(f"sqlite:///{(DATA / 'mlflow.db').resolve()}")
if mlflow.get_experiment_by_name('chatbot-qa') is None:
    mlflow.create_experiment('chatbot-qa', artifact_location=(DATA / 'mlartifacts').resolve().as_uri())
mlflow.set_experiment('chatbot-qa')

ANSWER_PROMPT = '''You are the customer-support assistant for an eBay store that sells motorcycle and small-engine parts.
Answer the buyer using only the store information below. If it does not cover the question, say a team member will follow up.
Store information:
{context}'''

JUDGE_VERSION = 'v2'
JUDGE_SYSTEM = f'''You grade replies from the customer-support chatbot of an eBay motorcycle and small-engine parts store.
Store facts:
{FACTS_TEXT}

You get a buyer question, the answer key points, the chatbot reply, and sometimes the listing the question is about.
- points: for each key point, in order, true if the reply covers it (same meaning, wording can differ), else false.
- escalated: true if the reply hands the buyer to a person or says staff will follow up or take action.
- unsupported_claim: true if the reply states a policy, time, price, fit or promise that contradicts or is not in the store facts or in the listing (when one is given).
- reason: one short sentence.
Reply with only JSON: {{"points": [...], "escalated": true or false, "unsupported_claim": true or false, "reason": "..."}}'''

def generate_answer(question, context):
    resp = client.models.generate_content(model=MODEL, contents=question,
        config={'system_instruction': ANSWER_PROMPT.format(context=context), 'temperature': 0})
    return resp.text

def collect(run, answer_fn):
    path = QA / f'replies_{run}.csv'
    done = pd.read_csv(path, dtype=str) if path.exists() else pd.DataFrame(columns=['qid', 'reply', 'latency_s'])
    todo = test[~test['qid'].isin(done['qid'])]
    print(f'{run}: {len(todo)} questions to answer')
    for i, r in enumerate(todo.itertuples(), 1):
        t0 = time.time()
        try:
            reply = answer_fn(r.question)
        except Exception as e:
            print('\n  stopped:', str(e)[:80], '- run this cell again later'); return False
        done = pd.concat([done, pd.DataFrame([{'qid': r.qid, 'reply': reply,
                                               'latency_s': round(time.time() - t0, 2)}])], ignore_index=True)
        done.to_csv(path, index=False)
        print(f'  answered {i}/{len(todo)}', end='\r'); time.sleep(PAUSE)
    return True

def grade(run):
    path = QA / f'graded_{run}.csv'
    replies = pd.read_csv(QA / f'replies_{run}.csv', dtype=str).fillna('')
    old = pd.read_csv(path, dtype={'qid': str}) if path.exists() else pd.DataFrame(columns=['qid'])
    rows, seen = old.to_dict('records'), set(old['qid'])
    todo = [r for r in test.merge(replies, on='qid').itertuples() if r.qid not in seen]
    print(f'{run}: {len(todo)} replies to grade')
    for i, r in enumerate(todo, 1):
        payload = {'question': r.question, 'key_points': r.key_points, 'reply': r.reply}
        if r.source == 'listing':
            payload['listing'] = r.source_text
        try:
            g = ask_gemini(JUDGE_SYSTEM, payload)
        except RuntimeError as e:
            print('\n ', e); return None
        pts = [bool(p) for p in g.get('points', [])][:len(r.key_points)]
        pts += [False] * (len(r.key_points) - len(pts))
        rows.append({'qid': r.qid, 'points_covered': sum(pts), 'points_total': len(pts),
                     'escalated': bool(g.get('escalated')), 'unsupported_claim': bool(g.get('unsupported_claim')),
                     'reason': g.get('reason', '')})
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f'  graded {i}/{len(todo)}', end='\r'); time.sleep(PAUSE)
    return score(run)

def score(run):
    gr = (test.merge(pd.read_csv(QA / f'replies_{run}.csv', dtype=str).fillna(''), on='qid')
              .merge(pd.read_csv(QA / f'graded_{run}.csv', dtype={'qid': str}), on='qid'))
    for col in ['escalated', 'unsupported_claim']:
        gr[col] = gr[col].astype(str).str.lower() == 'true'
    gr['escalation_ok'] = gr['escalated'] == gr['escalate']
    gr['correct'] = ((gr['points_covered'] == gr['points_total']) & gr['escalation_ok']
                     & ~gr['unsupported_claim'] & (gr['reply'] != ''))
    answerable = gr[~gr['escalate']]
    metrics = {'reply_accuracy': gr['correct'].mean(),
               'automation_rate': (answerable['correct'] & ~answerable['escalated']).mean(),
               'key_point_coverage': gr['points_covered'].sum() / gr['points_total'].sum(),
               'escalation_accuracy': gr['escalation_ok'].mean(),
               'unsupported_rate': gr['unsupported_claim'].mean(),
               'median_latency_s': pd.to_numeric(gr['latency_s']).median()}
    return gr, metrics

def run_eval(run, answer_fn, params):
    if not collect(run, answer_fn):
        return
    result = grade(run)
    if result is None:
        return
    gr, m = result
    print(pd.Series(m).round(3).to_string())
    print('\nReply accuracy by category:\n', gr.groupby('category')['correct'].agg(['mean', 'count']).round(2).to_string())
    with mlflow.start_run(run_name=run):
        mlflow.log_params({'test_set': 'qa_test_set_v1.csv', 'test_size': len(test), 'labels_source': LABELS_SOURCE,
                           'judge_model': MODEL, 'judge_version': JUDGE_VERSION, **params})
        mlflow.log_metrics({k: round(float(v), 4) for k, v in m.items()})
        for f in [QA / f'graded_{run}.csv', QA / 'qa_test_set_v1.csv', QA / 'store_facts_v1.json']:
            mlflow.log_artifact(str(f))
    print(f'\nLogged {run} to MLflow (browse: mlflow ui --backend-store-uri data/mlruns)')

### Run 1: all facts in the prompt

In [ ]:
run_eval('facts-in-prompt', lambda q: generate_answer(q, FACTS_TEXT),
         {'answer_model': MODEL, 'knowledge': 'store facts', 'retrieval': 'none'})

**Spot-check the judge.** The judge is an LLM too, so read 10 of its grades and count how many you agree with.

In [ ]:
pd.set_option('display.max_colwidth', 250)
spot = score('facts-in-prompt')[0].sample(10, random_state=0)
spot[['question', 'key_points', 'reply', 'escalate', 'escalated', 'unsupported_claim', 'correct', 'reason']]

In [ ]:
JUDGE_AGREE = 8   # TODO: how many of the 10 grades above you agree with
assert JUDGE_AGREE is not None, 'Read the 10 grades above and set JUDGE_AGREE'
(QA / 'judge_spotcheck.json').write_text(json.dumps({'run': 'facts-in-prompt', 'agree': JUDGE_AGREE, 'checked': 10}))
print(f'Judge agreement: {JUDGE_AGREE}/10')

### Run 2: RAG

Each store fact and each listing is one chunk. Chunks and questions are embedded in batches with `gemini-embedding-001` (saved to `cache/`), and the 4 closest chunks go into the prompt.

In [ ]:
TOP_K = 4
chunks = [f'{k.replace("_", " ")}: {v}' for k, v in FACTS.items()] + listings['text'].tolist()

def embed(texts, task, cache_name, batch=20):
    cached = load_cache(cache_name)
    todo = [t for t in dict.fromkeys(texts) if t not in cached]
    for start in range(0, len(todo), batch):
        part = todo[start:start + batch]
        for attempt in range(4):
            try:
                res = client.models.embed_content(model=EMBED_MODEL, contents=part,
                                                  config=types.EmbedContentConfig(task_type=task))
                break
            except Exception as e:
                wait = 60 * (attempt + 1)   # 429 means the per-minute quota: wait at least a minute
                print(f'\n  embedding attempt {attempt + 1} failed ({str(e)[:60]}), retrying in {wait}s')
                time.sleep(wait)
        else:
            raise RuntimeError('Embedding failed; run this cell again later (finished batches are cached)')
        cached.update({t: e.values for t, e in zip(part, res.embeddings)})
        save_cache(cache_name, cached)
        print(f'  embedded {min(start + batch, len(todo))}/{len(todo)}', end='\r')
        time.sleep(10)              # pause between batches to stay under the per-minute limit
    v = np.array([cached[t] for t in texts], dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

kb_vecs = embed(chunks, 'RETRIEVAL_DOCUMENT', 'embeddings_kb')
q_vecs = dict(zip(test['question'], embed(test['question'].tolist(), 'RETRIEVAL_QUERY', 'embeddings_questions')))
print(f'\nKnowledge base: {len(chunks)} chunks ({len(listings)} listings), {kb_vecs.nbytes / 1e6:.1f} MB of vectors')

def retrieve(question, k=TOP_K):
    v = q_vecs.get(question)
    if v is None:
        v = embed([question], 'RETRIEVAL_QUERY', 'embeddings_questions')[0]
    return [chunks[i] for i in np.argsort(-(kb_vecs @ v))[:k]]

for q in test[test['source'] == 'listing']['question'].head(2):   # check retrieval before spending requests
    print('\nQ:', q)
    for c in retrieve(q):
        print('   ->', c[:110])

In [ ]:
run_eval('rag', lambda q: generate_answer(q, '\n'.join(f'- {c}' for c in retrieve(q))),
         {'answer_model': MODEL, 'knowledge': f'store facts + {len(listings)} listings',
          'retrieval': EMBED_MODEL, 'top_k': TOP_K})

In [ ]:
# For each listing question, is the listing it was written from among the top k retrieved?
listing_qs = test[test['source'] == 'listing']
listing_vecs = kb_vecs[len(FACTS):]
listing_texts = chunks[len(FACTS):]

def rank_of_source(row):
    v = q_vecs[row['question']]
    order = np.argsort(-(listing_vecs @ v))
    # the source listing text was cut to 1,200 characters, so compare the first 100
    hits = [i for i, idx in enumerate(order) if listing_texts[idx][:100] == row['source_text'][:100]]
    return hits[0] + 1 if hits else None

ranks = listing_qs.apply(rank_of_source, axis=1)
for k in [1, 3, 4, 10]:
    print(f'Recall@{k}: {(ranks <= k).mean():.0%}  (source listing in the top {k})')
print('\nQuestions where the source listing was not in the top 4:')
print(listing_qs.loc[ranks.isna() | (ranks > 4), 'question'].to_string())

In [ ]:
# Did the answer go wrong because retrieval missed the listing, or despite finding it?
rag_graded = score('rag')[0].set_index('qid')
check = listing_qs[['qid', 'question']].copy()
check['rank'] = ranks.values
check['found_in_top4'] = check['rank'] <= 4
check['answer_correct'] = check['qid'].map(rag_graded['correct'])

print(pd.crosstab(check['found_in_top4'], check['answer_correct'],
                  rownames=['listing found in top 4'], colnames=['answer correct']))

In [ ]:
pd.set_option('display.max_colwidth', 200)
bad = check[check['found_in_top4'] & (check['answer_correct'] == False)]['qid']
g = rag_graded.loc[bad]
print('Escalated when it should not:', (g['escalated'] & ~g['escalate']).sum(), 'of', len(g))
print('Missed key points:', (g['points_covered'] < g['points_total']).sum(), 'of', len(g))
g[['question', 'key_points', 'reply', 'escalated', 'reason']]

In [ ]:
listing_vecs = kb_vecs[len(FACTS):]             # the listings part of the knowledge base
listing_chunks = chunks[len(FACTS):]

def retrieve_listings(question, k=3):
    v = q_vecs.get(question)
    if v is None:
        v = embed([question], 'RETRIEVAL_QUERY', 'embeddings_questions')[0]
    return [listing_chunks[i] for i in np.argsort(-(listing_vecs @ v))[:k]]

print('retrieve_listings ready; example:', retrieve_listings('Will this fit my 2001 Yamaha TTR225?')[0][:80])

In [ ]:
ANSWER_PROMPT_V2 = '''You are the customer-support assistant for an eBay store that sells motorcycle and small-engine parts.
Answer the buyer using only the store information below. Never promise anything that is not in it.
Rules:
- Fit questions: say what the listing states about fit, then remind the buyer to match the OEM part number or send their year, make and model so we can double-check before they buy.
- Problems only staff can solve (refund, return, replacement, wrong, damaged, missing or late item): explain the relevant policy, ask for photos or order details if useful, and say a team member will follow up.
- If the store information does not cover the question, say a team member will follow up.
Store information:
{context}'''

def answer_v2(question, context):
    resp = client.models.generate_content(model=MODEL, contents=question,
        config={'system_instruction': ANSWER_PROMPT_V2.format(context=context), 'temperature': 0})
    return resp.text

def answer_facts_plus_listings_v2(question):
    listings_text = '\n'.join(f'- {c}' for c in retrieve_listings(question))
    return answer_v2(question, f'{FACTS_TEXT}\n\nRelevant listings:\n{listings_text}')

run_eval('facts-plus-listing-rag-prompt-v2', answer_facts_plus_listings_v2,
         {'answer_model': MODEL, 'answer_prompt': 'v2', 'knowledge': f'all store facts + top 3 of {len(listings)} listings',
          'retrieval': EMBED_MODEL, 'top_k': 3})

In [ ]:
gr, m = score('facts-plus-listing-rag-prompt-v2')
print(len(gr), 'of 70 graded so far (preliminary)')
print(pd.Series(m).round(3).to_string())
print(gr.groupby('category')['correct'].agg(['mean', 'count']).round(2).to_string())

### Optional: the real chatbot

Start the Flask backend in another terminal and fill in the chat route from `backend/app.py`. Only fair once the store facts match the policies the app uses.

In [ ]:
import requests

CHATBOT_URL = 'http://127.0.0.1:5050/api/support/chat'
TROUBLE = "Sorry, I'm having trouble answering right now"

def answer_real_chatbot(question):
    r = requests.post(CHATBOT_URL, json={'conversation_id': None, 'message': question}, timeout=120)
    if r.status_code == 429:
        raise RuntimeError('Chatbot rate limit (12 messages a minute); run again in a minute')
    r.raise_for_status()
    d = r.json()
    reply = d.get('reply') or ''
    if d.get('mode') != 'ai' or reply.startswith(TROUBLE):
        raise RuntimeError(f"Chatbot did not answer with AI (mode={d.get('mode')}): {reply[:80]}")
    if d.get('needs_human'):
        reply += '\n\n[The conversation was handed to a person on the team.]'
    return reply

PAUSE_BEFORE = PAUSE
PAUSE = 6          # the chatbot allows 12 messages a minute per visitor
try:
    run_eval('baseline-real-chatbot', answer_real_chatbot,
             {'answer_model': 'app (gemini-3.1-flash-lite)', 'knowledge': 'app articles + product search tool',
              'retrieval': 'app tools'})
finally:
    PAUSE = PAUSE_BEFORE

In [ ]:
real = pd.read_csv(QA / 'replies_baseline-real-chatbot.csv', dtype=str).fillna('')
real = test.merge(real, on='qid')
real['handed_off'] = real['reply'].str.contains('handed to a person', regex=False)

print('Handed to staff, by question type (all after-sale questions should be):')
print(real.groupby('source')['handed_off'].agg(['sum', 'count']).to_string())
print('\nEscalation right or wrong (preliminary, no judge):', f"{(real['handed_off'] == real['escalate']).mean():.0%}")

## 6. Compare

In [ ]:
runs = {r: score(r) for r in ['baseline-real-chatbot', 'facts-in-prompt', 'rag',
                              'facts-plus-listing-rag', 'facts-plus-listing-rag-prompt-v2']
        if (QA / f'graded_{r}.csv').exists()}
print(pd.DataFrame({r: m for r, (_, m) in runs.items()}).T.round(3).to_string())
print('\nReply accuracy by question type:')
print(pd.DataFrame({r: g.groupby('source')['correct'].mean() for r, (g, _) in runs.items()}).round(2).to_string())
spot = QA / 'judge_spotcheck.json'
if spot.exists():
    s = json.loads(spot.read_text())
    print(f"\nJudge agreement (spot-check): {s['agree']}/{s['checked']}")
print(f"Test set: {len(test)} questions {test['source'].value_counts().to_dict()}; knowledge base: {len(listings)} listings")

Questions RAG got wrong. If retrieval missed the right chunk, the fix is the knowledge base; if it found it but answered badly, the fix is the prompt.

In [ ]:
if 'rag' in runs:
    wrong = runs['rag'][0]
    wrong = wrong[~wrong['correct']].copy()
    wrong['retrieved'] = wrong['question'].map(lambda q: ' || '.join(c[:60] for c in retrieve(q)))
    display(wrong[['source', 'question', 'retrieved', 'reply', 'reason']])